In [0]:
%sql

create or replace table retailsales.gold.daily_store_sales as
select a.store_id,b.store_name,b.state,b.city,b.region,b.store_type,b.country, sum(quantity*unit_price) -sum(discount_amount) as total_sales,sum(quantity) quantity,count(distinct a.sales_id) transactions,count(distinct a.customer_id) customers,count(distinct a.product_id) products,sum(quantity*unit_price) -sum(discount_amount) / count(distinct a.sales_id) as avg_sales_per_transaction , current_timestamp() as insert_ts,date(created_timestamp) transaction_date
 from retailsales.silver.sales a
left join retailsales.silver.dim_store b 
on a.store_id=b.store_id
--and a.created_timestamp between b.effective_from and coalesce(b.--effective_to,'9999-12-31')
group by a.store_id,b.store_name,b.state,b.city,b.region,b.store_type,b.country,date(created_timestamp) 

In [0]:
%sql
select * from retailsales.gold.daily_store_sales 

In [0]:
%sql
ALTER TABLE retailsales.silver.sales
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
DECLARE OR REPLACE VARIABLE current_version BIGINT;
SET VAR current_version = (
    SELECT COALESCE(MAX(version), 4)
    FROM retailsales.gold.latest_version_gold_table
    WHERE tbl_nm = 'daily_store_sales'
);

In [0]:
%sql

create or replace table retailsales.silver.store_sales_updates as
select a.store_id,b.store_name,b.state,b.city,b.region,b.store_type,b.country, sum(quantity*unit_price) -sum(discount_amount) as total_sales,sum(quantity) quantity,count(distinct a.sales_id) transactions,count(distinct a.customer_id) customers,count(distinct a.product_id) products,sum(quantity*unit_price) -sum(discount_amount) / count(distinct a.sales_id) as avg_sales_per_transaction , current_timestamp() as insert_ts,date(created_timestamp) transaction_date
 from retailsales.silver.sales a
left join retailsales.silver.dim_store b 
on a.store_id=b.store_id
inner join(select a.store_id,date(created_timestamp) transaction_date  from table_changes('retailsales.silver.sales',current_version)a
group by a.store_id,date(created_timestamp) )c 
on a.store_id=c.store_id and date(a.created_timestamp) =c.transaction_date
--and a.created_timestamp between b.effective_from and coalesce(b.--effective_to,'9999-12-31')
group by a.store_id,b.store_name,b.state,b.city,b.region,b.store_type,b.country,date(created_timestamp) 



In [0]:
%sql
merge into retailsales.gold.daily_store_sales  target
using retailsales.silver.store_sales_updates source
on target.store_id = source.store_id and target.transaction_date = source.transaction_date
when matched then update set *
when not matched then insert *


In [0]:
%sql
insert into retailsales.gold.latest_version_gold_table 
select max(_commit_version) as version, current_timestamp() as ts,'daily_store_sales' tbl_nm from table_changes('retailsales.silver.sales',4)